In [3]:
import os
from pathlib import Path

# Check Kaggle input directory for datasets
input_dir = Path('/kaggle/input')
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_images(root: Path, limit=None):
    """Count image files recursively under root."""
    count = 0
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in image_exts:
            count += 1
            if limit is not None and count >= limit:
                return count
    return count


print('Datasets found in /kaggle/input:')
dataset_roots = []
for d in sorted(input_dir.iterdir()):
    if not d.is_dir():
        continue
    total_imgs = count_images(d)
    dataset_roots.append((d, total_imgs))
    print(f'- {d.name}: {total_imgs} images')

# Optional: set this manually if you want a specific dataset.
# Examples based on your screenshots:
#   /kaggle/input/celebahq-256-images-only
#   /kaggle/input/wikiart
#   /kaggle/input/ffhq-face-data-set
FORCE_DATASET = None

if FORCE_DATASET:
    DATA_DIR = FORCE_DATASET
else:
    non_empty = [(p, n) for p, n in dataset_roots if n > 0]
    DATA_DIR = str(max(non_empty, key=lambda x: x[1])[0]) if non_empty else None

print('Selected DATA_DIR:', DATA_DIR)
if DATA_DIR is None:
    raise ValueError('No images found in /kaggle/input. Check dataset attachment and path.')

Datasets found in /kaggle/input:
- datasets: 181446 images
Selected DATA_DIR: /kaggle/input/datasets


In [4]:
import os
import random
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data._utils.collate import default_collate
from PIL import Image, ImageFile, UnidentifiedImageError

# Allow loading slightly damaged images instead of crashing.
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMG_SIZE = 128
BATCH_SIZE = 16

# Time-budget controls (based on your measured speed).
TARGET_TOTAL_HOURS = 2.0
PLANNED_EPOCHS = 30
MEASURED_IT_PER_SEC = 1.25
SAFETY_FACTOR = 0.85  # keep margin for logging/checkpoint overhead


def is_valid_image(path):
    """Fast integrity check for image files."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class ImageDataset(Dataset):
    def __init__(self, folder, transform=None):
        if folder is None:
            raise ValueError('DATA_DIR is None. Set a valid dataset path.')

        exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        raw_files = []

        # Recursive scan so nested structures work:
        # - celebahq256_imgs/train/*.jpg
        # - wikiart/wikiart/<style>/*.jpg
        # - FFHQ/thumbnails128×128/*.png
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(exts):
                    raw_files.append(os.path.join(root, f))

        raw_files.sort()
        self.transform = transform

        # Remove corrupted files once, so workers don't crash during training.
        self.files = [p for p in raw_files if is_valid_image(p)]
        skipped = len(raw_files) - len(self.files)

        if len(self.files) == 0:
            raise ValueError(
                f'No valid image files found under {folder}. '
                'Check DATA_DIR or file integrity.'
            )

        print(f'Valid images: {len(self.files)} | Skipped corrupted: {skipped}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Return None on rare runtime decode failures; collate_fn will drop it.
        try:
            path = self.files[idx]
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img
        except (OSError, UnidentifiedImageError, ValueError):
            return None


def safe_collate(batch):
    batch = [x for x in batch if x is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

full_dataset = ImageDataset(DATA_DIR, transform)

# Compute target steps/epoch and target sample count to fit 2-hour total budget.
seconds_per_epoch_budget = (TARGET_TOTAL_HOURS * 3600.0) / PLANNED_EPOCHS
TARGET_STEPS_PER_EPOCH = max(1, int(seconds_per_epoch_budget * MEASURED_IT_PER_SEC * SAFETY_FACTOR))
target_samples = TARGET_STEPS_PER_EPOCH * BATCH_SIZE

if len(full_dataset) > target_samples:
    rng = random.Random(42)
    indices = list(range(len(full_dataset)))
    rng.shuffle(indices)
    indices = indices[:target_samples]
    dataset = Subset(full_dataset, indices)
else:
    dataset = full_dataset

# Fewer workers are often more stable in Kaggle with heavy image decode.
NUM_WORKERS = 2

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=safe_collate,
    drop_last=True
)

print('Using reduced dataset for time budget.')
print('Planned epochs:', PLANNED_EPOCHS)
print('Target total hours:', TARGET_TOTAL_HOURS)
print('Measured it/s:', MEASURED_IT_PER_SEC)
print('Target steps/epoch:', TARGET_STEPS_PER_EPOCH)
print('Target sample count:', target_samples)
print('Actual samples used:', len(dataset))
print('Actual batches/epoch:', len(dataloader))

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Valid images: 181446 | Skipped corrupted: 0
Using reduced dataset for time budget.
Planned epochs: 30
Target total hours: 2.0
Measured it/s: 1.25
Target steps/epoch: 255
Target sample count: 4080
Actual samples used: 4080
Actual batches/epoch: 255


In [ ]:
import torch
import matplotlib.pyplot as plt

T = 300  # timesteps

def linear_schedule(T):
    beta_start = 1e-4
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, T)

betas = linear_schedule(T)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod)
sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod)

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    s1 = sqrt_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    s2 = sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    return s1 * x0 + s2 * noise, noise

# Visualize 5 forward diffusion steps
sample_img = next(iter(dataloader))[0].unsqueeze(0)  # single image
steps = [0, 60, 120, 180, 299]

fig, axes = plt.subplots(1, len(steps), figsize=(15, 3))
for i, step in enumerate(steps):
    t = torch.tensor([step])
    noisy, _ = q_sample(sample_img, t)
    img_np = noisy[0].permute(1, 2, 0).numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    axes[i].imshow(img_np)
    axes[i].set_title(f't={step}')
    axes[i].axis('off')
plt.suptitle('Forward Diffusion Steps')
plt.tight_layout()
plt.savefig('/kaggle/working/forward_diffusion.png', dpi=100)
plt.show()

In [ ]:
import torch
import torch.nn as nn
import math

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2 = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, time_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, t):
        x = self.res(x, t)
        return self.down(x), x

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.res = ResBlock(in_ch + skip_ch, out_ch, time_dim)

    def forward(self, x, skip, t):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x, t)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_emb = TimeEmbedding(time_dim)
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)

        self.mid = ResBlock(base_ch * 4, base_ch * 4, time_dim)

        self.up1 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up2 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)

        self.out = nn.Sequential(nn.GroupNorm(8, base_ch), nn.SiLU(), nn.Conv2d(base_ch, in_ch, 1))

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x = self.init_conv(x)
        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x = self.mid(x, t_emb)
        x = self.up1(x, s2, t_emb)
        x = self.up2(x, s1, t_emb)
        return self.out(x)


def unwrap_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print('Device:', device)
print('GPU count:', gpu_count)
if gpu_count > 0:
    for i in range(gpu_count):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

model = UNet()
if gpu_count > 1:
    print('Using multi-GPU training with nn.DataParallel')
    model = nn.DataParallel(model)

model = model.to(device)
total_params = sum(p.numel() for p in unwrap_model(model).parameters())
print('Parameters:', total_params)

In [ ]:
import torch
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm
import time

EPOCHS = 30
LR = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = (device.type == 'cuda')
scaler = GradScaler('cuda', enabled=use_amp)

# move noise schedule to device
sqrt_alpha_cumprod = sqrt_alpha_cumprod.to(device)
sqrt_one_minus_alpha_cumprod = sqrt_one_minus_alpha_cumprod.to(device)

def q_sample_device(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    s1 = sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
    s2 = sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
    return s1 * x0 + s2 * noise, noise

def format_time(seconds):
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h:02d}:{m:02d}:{s:02d}'
    return f'{m:02d}:{s:02d}'

# Respect target steps from preprocessing cell if present.
max_steps_per_epoch = globals().get('TARGET_STEPS_PER_EPOCH', len(dataloader))

loss_history = []
train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0
    epoch_start = time.time()

    pbar = tqdm(dataloader, total=min(len(dataloader), max_steps_per_epoch), desc=f'Epoch {epoch}/{EPOCHS}', leave=True)
    for step, imgs in enumerate(pbar, start=1):
        if step > max_steps_per_epoch:
            break

        if imgs is None:
            continue

        imgs = imgs.to(device, non_blocking=True)
        t = torch.randint(0, T, (imgs.size(0),), device=device)
        noise = torch.randn_like(imgs)
        noisy_imgs, noise = q_sample_device(imgs, t, noise)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=use_amp):
            pred = model(noisy_imgs, t)
            loss = F.mse_loss(pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches += 1

        avg_so_far = total_loss / max(1, num_batches)
        pbar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{avg_so_far:.4f}')

    if num_batches == 0:
        raise RuntimeError('All batches were empty after filtering corrupted images.')

    avg = total_loss / num_batches
    loss_history.append(avg)

    epoch_time = time.time() - epoch_start
    elapsed_total = time.time() - train_start
    avg_epoch_time = elapsed_total / epoch
    eta_total = avg_epoch_time * (EPOCHS - epoch)

    print(
        f'Epoch {epoch}/{EPOCHS} complete | '
        f'Steps: {num_batches}/{max_steps_per_epoch} | '
        f'Loss: {avg:.4f} | '
        f'Epoch Time: {format_time(epoch_time)} | '
        f'ETA Remaining: {format_time(eta_total)}'
    )

model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
torch.save(model_to_save.state_dict(), '/kaggle/working/ddpm_model.pth')
print('Model saved.')